# Regional Macro Data-Quality Audit

## tl;dr

This companion notebook validates the currently promoted dashboard vintage. Its summary is generated from executed outputs below; warnings are retained rather than silently repaired.

## Context & Methods

The intended grain is region × source period × industry/indicator/revenue code × source vintage. The audit checks canonical columns, unique grain, required values, domain rules, FNS period-flow logic, Treasury accounting reconciliation, and visible source gaps.

### Key Assumptions

- FNS 1-NOM is cumulative YTD and reported in thousand rubles; promoted values are rubles.
- A period flow is derived only when the immediately preceding source period exists; Q1 YTD equals Q1 flow.
- Treasury approved plan is the plan reported at the cutoff, not necessarily the originally enacted plan.
- Rosstat indices are published percentages relative to the same month a year earlier; values are not rebased or spliced into levels.

## Data

### 1. Load the promoted vintage and manifest

In [1]:
from pathlib import Path
import json, sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from macro_rus.contracts import TABLE_FILES
from macro_rus.validation import validate_tables

DATA_DIR = PROJECT_ROOT / 'data' / 'promoted' / 'current'
manifest = json.loads((DATA_DIR / 'manifest.json').read_text(encoding='utf-8'))
tables = {name: pd.read_parquet(DATA_DIR / filename) for name, filename in TABLE_FILES.items()}
pd.DataFrame(manifest['tables']).T

,rows,coverage_start,coverage_end
pit_receipts,80,2025-12-31T00:00:00,2026-04-01T00:00:00
industrial_production,1390,2015-01-31T00:00:00,2026-07-31T00:00:00
budget_execution,20,2026-03-31T00:00:00,2026-06-30T00:00:00
regions,2.0,NaN,NaN
sources,6,2022-09-30T00:00:00,2026-07-31T00:00:00
quality_events,59,2026-03-31T00:00:00,2026-06-30T00:00:00


## Results

### 2. Re-run canonical contract and grain checks

In [2]:
validation = validate_tables({k: v for k, v in tables.items() if k != 'quality_events'})
summary = (validation.events.groupby(['severity','status']).size().rename('checks').reset_index())
summary

,severity,status,checks
0,critical,passed,10
1,high,passed,31
2,medium,passed,5
3,medium,warning,1


### 3. Review retained warnings

In [3]:
quality = tables['quality_events']
warnings = quality[quality['status'].isin(['warning','failed'])][['severity','table_name','region_id','period','message','observed_value']]
warnings.reset_index(drop=True)

,severity,table_name,region_id,period,message,observed_value
0,medium,pit_receipts,None,NaT,3 negative reported YTD values retained for re...,3
1,medium,budget_execution,None,NaT,An unused ancillary member has a CRC failure; ...,Доходы_расходы_источники по графам на 01.04.20...
2,medium,budget_execution,None,NaT,An unused ancillary member has a CRC failure; ...,Доходы_расходы_источники по графам на 01.07.20...
3,medium,pit_receipts,RU-KLU,2026-03-31,FNS collected PIT and Treasury retained consol...,0.05203739991241813
4,medium,pit_receipts,None,NaT,"65 FNS downloads were not ingested, including ...",None
5,low,industrial_production,None,NaT,2 non-monthly or duplicate Rosstat workbooks w...,None


### 4. Reconcile Q1 FNS flows to reported YTD

In [4]:
pit = tables['pit_receipts'].copy()
pit['period'] = pd.to_datetime(pit['period'])
q1 = pit[pit['period'].dt.quarter.eq(1)]
q1_flow_gap = (q1['pit_flow_rub'] - q1['pit_ytd_rub']).abs()
pd.DataFrame({'q1_rows': [len(q1)], 'max_abs_flow_gap_rub': [q1_flow_gap.max()], 'negative_ytd_rows': [(pit['pit_ytd_rub'] < 0).sum()]})

,q1_rows,max_abs_flow_gap_rub,negative_ytd_rows
0,40,0.0,3


### 5. Reconcile Treasury total revenue

In [5]:
budget = tables['budget_execution'].copy()
wide = budget.pivot_table(index=['region_id','period'], columns='revenue_code', values='actual_ytd_rub', aggfunc='sum')
wide['reconciliation_gap_rub'] = wide['TOTAL_REVENUE'] - wide['10000000000000000'] - wide['20000000000000000']
wide[['TOTAL_REVENUE','10000000000000000','20000000000000000','reconciliation_gap_rub']]

revenue_code          TOTAL_REVENUE  10000000000000000  20000000000000000  \
region_id period                                                            
RU-KLU    2026-03-31   3.029749e+10       2.742327e+10       2.874222e+09   
          2026-06-30   7.216411e+10       6.390081e+10       8.263303e+09   
RU-SVE    2026-03-31   1.308444e+11       1.212038e+11       9.640576e+09   
          2026-06-30   2.804942e+11       2.616984e+11       1.879584e+10   

revenue_code          reconciliation_gap_rub  
region_id period                              
RU-KLU    2026-03-31                0.000000  
          2026-06-30                0.000006  
RU-SVE    2026-03-31               -0.000004  
          2026-06-30                0.000023

### 6. Confirm coverage and source-vintage traceability

In [6]:
coverage=[]
for name, frame in tables.items():
    if name in {'quality_events','regions','sources'}: continue
    periods=pd.to_datetime(frame['period'],errors='coerce')
    coverage.append({'table':name,'rows':len(frame),'regions':frame['region_id'].nunique(),'start':periods.min(),'end':periods.max(),'source_vintages':frame['source_vintage'].nunique()})
pd.DataFrame(coverage)

,table,rows,regions,start,end,source_vintages
0,pit_receipts,80,2,2025-12-31,2026-03-31,2
1,industrial_production,1390,2,2015-01-31,2026-07-31,2
2,budget_execution,20,2,2026-03-31,2026-06-30,2


## Takeaways

- The promoted pilot passes blocking contract, uniqueness, required-value, and Treasury accounting checks.
- The Rosstat monthly history is sufficient for the production cross-check.
- FNS history remains incomplete because most historical releases do not yet have their exact matching schema files. The dashboard therefore suppresses unsupported PIT year-on-year contribution claims.
- Negative reported FNS adjustments are retained and visible for analyst review.
- Treasury execution is descriptive until historical seasonal packages and original budget-law annexes are added.